# Agentes Inteligentes

## Contexto e Motivação

Este notebook foi reorganizado como um tutorial sobre a evolução de agentes no problema do aspirador. A ideia é começar com uma estratégia puramente tabelada, avançar para um agente reflexo e terminar em um agente baseado em modelo.

Essa progressão é importante porque mostra um ganho conceitual claro: primeiro o agente depende de uma tabela explícita de histórico de percepções, depois passa a reagir apenas ao percepto atual, e por fim incorpora memória interna para tomar decisões melhores.


## Setup

As células abaixo preparam o ambiente do notebook. A instalação de `ipythonblocks` é opcional neste tutorial, porque o foco não está em visualização gráfica.


In [ ]:
# Descomente a linha abaixo apenas se o seu ambiente precisar da dependência.
# %pip install -q ipythonblocks


In [ ]:
import random
import sys
import types

try:
    from ipythonblocks import BlockGrid
except ModuleNotFoundError:
    modulo = types.ModuleType('ipythonblocks')

    class BlockGrid(list):
        def __init__(self, *args, **kwargs):
            self.width = args[0] if args else None
            self.height = args[1] if len(args) > 1 else None

        def show(self):
            return None

    modulo.BlockGrid = BlockGrid
    sys.modules['ipythonblocks'] = modulo

from agents import *


## Fundamentação Teórica

### Agente como função

Um agente pode ser visto como uma função que transforma sequências de percepções em ações:

$$
\text{agent}: P^* \rightarrow A
$$

em que $P^*$ representa todas as sequências possíveis de percepções e $A$ representa o conjunto de ações disponíveis.

### Medida de desempenho

Para avaliar um agente, usamos uma medida acumulada de recompensas e penalidades:

$$
J(\pi) = \sum_{t=0}^{T} r_t
$$

em que $\pi$ é a política do agente e $r_t$ representa o ganho ou custo no instante $t$.

### Três níveis de agente neste tutorial

- **Table-driven**: escolhe ações olhando para todo o histórico de percepções.
- **Reflex**: escolhe ações apenas com base no percepto atual.
- **Model-based**: usa o percepto atual e um estado interno que resume o que já aprendeu sobre o ambiente.


## Implementação Prática

O problema será traduzido para código usando o ambiente trivial do aspirador, com duas posições possíveis: `loc_A` e `loc_B`. Antes dos exemplos, definimos uma função obrigatória que converte saídas técnicas em explicações em português.


In [ ]:
def traduzir_saida_tecnica_em_portugues(resultado):
    if isinstance(resultado, tuple) and len(resultado) == 3:
        estado, posicao, desempenho = resultado
        if isinstance(estado, dict):
            estado_texto = ", ".join(f"{k}: {v}" for k, v in estado.items())
        else:
            estado_texto = str(estado)
        return (
            f"O agente terminou na posição {posicao}, com desempenho {desempenho}. "
            f"No final, o ambiente ficou assim: {estado_texto}."
        )
    return f"Explicação em português: {resultado}"


### Classe `Thing`

Antes dos agentes do aspirador, precisamos da abstração mais básica do ambiente: `Thing`. Ela representa qualquer entidade que possa existir no mundo.


In [ ]:
class Thing:
    def __repr__(self):
        return '<{}>'.format(getattr(self, '__name__', self.__class__.__name__))

    def is_alive(self):
        return hasattr(self, 'alive') and self.alive

    def show_state(self):
        print("I don't know how to show_state.")

    def display(self, canvas, x, y, width, height):
        pass


### Classe `Agent`

Agora vem `Agent`, que herda de `Thing` e adiciona o programa de decisão e o estado interno mínimo necessário para a simulação.


In [ ]:
import collections

class Agent(Thing):
    def __init__(self, program=None):
        self.alive = True
        self.bump = False
        self.holding = []
        self.performance = 0
        if program is None or not isinstance(program, collections.abc.Callable):
            print("Can't find a valid program for {}, falling back to default.".format(self.__class__.__name__))

            def program(percept):
                return eval(input('Percept={}; action? '.format(percept)))

        self.program = program

    def can_grab(self, thing):
        return False


### Classe `Environment`

Em seguida, precisamos da estrutura do ambiente. O recorte abaixo já é suficiente para executar o tutorial: ele registra objetos, insere agentes, entrega perceptos e aplica ações passo a passo.


In [ ]:
class Environment:
    def __init__(self):
        self.things = []
        self.agents = []

    def percept(self, agent):
        raise NotImplementedError

    def execute_action(self, agent, action):
        raise NotImplementedError

    def default_location(self, thing):
        return None

    def exogenous_change(self):
        pass

    def is_done(self):
        return not any(agent.is_alive() for agent in self.agents)

    def step(self):
        if not self.is_done():
            actions = []
            for agent in self.agents:
                if agent.alive:
                    actions.append(agent.program(self.percept(agent)))
                else:
                    actions.append("")
            for (agent, action) in zip(self.agents, actions):
                self.execute_action(agent, action)
            self.exogenous_change()

    def run(self, steps=1000):
        for step in range(steps):
            if self.is_done():
                return
            self.step()

    def add_thing(self, thing, location=None):
        if location is None:
            location = self.default_location(thing)
        thing.location = location
        self.things.append(thing)
        if isinstance(thing, Agent):
            thing.performance = 0
            self.agents.append(thing)


### Classe `TrivialVacuumEnvironment`

Este é o ambiente usado em todo o tutorial. Ele possui apenas duas localizações e devolve ao agente uma percepção na forma `(posição, estado_do_local)`.


In [ ]:
class TrivialVacuumEnvironment(Environment):
    def __init__(self):
        super().__init__()
        self.status = {loc_A: random.choice(['Clean', 'Dirty']),
                       loc_B: random.choice(['Clean', 'Dirty'])}

    def percept(self, agent):
        return agent.location, self.status[agent.location]

    def execute_action(self, agent, action):
        if action == 'Right':
            agent.location = loc_B
            agent.performance -= 1
        elif action == 'Left':
            agent.location = loc_A
            agent.performance -= 1
        elif action == 'Suck':
            if self.status[agent.location] == 'Dirty':
                agent.performance += 10
            self.status[agent.location] = 'Clean'

## Decomposição de Processos

A partir daqui, o notebook alterna explicações e código para acompanhar a evolução do agente passo a passo.


### Etapa 1. Começar com um agente tabelado

O programa `TableDrivenAgentProgram` usa a sequência completa de percepções como chave de consulta em uma tabela. Isso é conceitualmente simples, mas pouco escalável, porque a tabela precisa prever muitos históricos possíveis.


In [ ]:
def TableDrivenAgentProgram(table):
    percepts = []

    def program(percept):
        percepts.append(percept)
        action = table.get(tuple(percepts))
        return action

    return program

In [2]:
tabela_exemplo = {
    ((loc_A, 'Dirty'),): 'Suck',
    ((loc_A, 'Dirty'), (loc_A, 'Clean')): 'Right',
    ((loc_A, 'Dirty'), (loc_A, 'Clean'), (loc_B, 'Dirty')): 'Suck'
}

programa_tabelado = TableDrivenAgentProgram(tabela_exemplo)
print(programa_tabelado((loc_A, 'Dirty')))
print(programa_tabelado((loc_A, 'Clean')))
print(programa_tabelado((loc_B, 'Dirty')))


NameError: name 'loc_A' is not defined

### Etapa 2. Encapsular a tabela em um agente do mundo do aspirador

A função `TableDrivenVacuumAgent` empacota essa ideia em um agente concreto para o ambiente trivial.


In [ ]:
def TableDrivenVacuumAgent():
    table = {((loc_A, 'Clean'),): 'Right',
             ((loc_A, 'Dirty'),): 'Suck',
             ((loc_B, 'Clean'),): 'Left',
             ((loc_B, 'Dirty'),): 'Suck',
             ((loc_A, 'Dirty'), (loc_A, 'Clean')): 'Right',
             ((loc_A, 'Clean'), (loc_B, 'Dirty')): 'Suck',
             ((loc_B, 'Clean'), (loc_A, 'Dirty')): 'Suck',
             ((loc_B, 'Dirty'), (loc_B, 'Clean')): 'Left',
             ((loc_A, 'Dirty'), (loc_A, 'Clean'), (loc_B, 'Dirty')): 'Suck',
             ((loc_B, 'Dirty'), (loc_B, 'Clean'), (loc_A, 'Dirty')): 'Suck'}
    return Agent(TableDrivenAgentProgram(table))

In [ ]:
ambiente_tabela = TrivialVacuumEnvironment()
ambiente_tabela.status = {loc_A: 'Dirty', loc_B: 'Dirty'}
agente_tabela = TraceAgent(TableDrivenVacuumAgent())
ambiente_tabela.add_thing(agente_tabela, loc_A)

for passo in range(4):
    print(f'Passo {passo + 1}')
    ambiente_tabela.step()
    print('Estado técnico:', ambiente_tabela.status, '| posição:', agente_tabela.location, '| desempenho:', agente_tabela.performance)

resultado_tabela = (dict(ambiente_tabela.status), agente_tabela.location, agente_tabela.performance)
print(traduzir_saida_tecnica_em_portugues(resultado_tabela))


### Etapa 3. Simplificar a decisão com um agente reflexo

O `ReflexVacuumAgent` abandona a tabela de históricos e reage apenas ao percepto atual. Isso reduz a complexidade da representação, desde que o percepto já contenha informação suficiente para decidir.


In [ ]:
def ReflexVacuumAgent():
    def program(percept):
        location, status = percept
        if status == 'Dirty':
            return 'Suck'
        elif location == loc_A:
            return 'Right'
        elif location == loc_B:
            return 'Left'

    return Agent(program)

In [ ]:
ambiente_reflexo = TrivialVacuumEnvironment()
ambiente_reflexo.status = {loc_A: 'Dirty', loc_B: 'Dirty'}
agente_reflexo = TraceAgent(ReflexVacuumAgent())
ambiente_reflexo.add_thing(agente_reflexo, loc_A)

for passo in range(4):
    print(f'Passo {passo + 1}')
    ambiente_reflexo.step()
    print('Estado técnico:', ambiente_reflexo.status, '| posição:', agente_reflexo.location, '| desempenho:', agente_reflexo.performance)

resultado_reflexo = (dict(ambiente_reflexo.status), agente_reflexo.location, agente_reflexo.performance)
print(traduzir_saida_tecnica_em_portugues(resultado_reflexo))


### Etapa 4. Adicionar memória com um agente baseado em modelo

O `ModelBasedVacuumAgent` mantém um pequeno estado interno com o que já sabe sobre `loc_A` e `loc_B`. Assim, ele consegue parar com `NoOp` quando conclui que todo o ambiente está limpo.


In [19]:
def ModelBasedVacuumAgent():
    model = {loc_A: None, loc_B: None}

    def program(percept):
        location, status = percept
        model[location] = status
        if model[loc_A] == model[loc_B] == 'Clean':
            return 'NoOp'
        elif status == 'Dirty':
            return 'Suck'
        elif location == loc_A:
            return 'Right'
        elif location == loc_B:
            return 'Left'

    return Agent(program)

### Etapa 4.1. Demonstração do `ModelBasedVacuumAgent`

Nesta execução, os dois lados começam limpos. O objetivo é observar que o agente precisa perceber o ambiente antes de concluir que pode parar com `NoOp`.


In [20]:
ambiente_modelo = TrivialVacuumEnvironment()
ambiente_modelo.status = {loc_A: 'Clean', loc_B: 'Clean'}
agente_modelo = TraceAgent(ModelBasedVacuumAgent())
ambiente_modelo.add_thing(agente_modelo, loc_A)

for passo in range(4):
    print(f'Passo {passo + 1}')
    ambiente_modelo.step()
    print('Estado técnico:', ambiente_modelo.status, '| posição:', agente_modelo.location, '| desempenho:', agente_modelo.performance)

resultado_modelo_demo = (dict(ambiente_modelo.status), agente_modelo.location, agente_modelo.performance)
print(traduzir_saida_tecnica_em_portugues(resultado_modelo_demo))

Passo 1
<Agent> perceives ((0, 0), 'Clean') and does Right
Estado técnico: {(0, 0): 'Clean', (1, 0): 'Clean'} | posição: (1, 0) | desempenho: -1
Passo 2
<Agent> perceives ((1, 0), 'Clean') and does NoOp
Estado técnico: {(0, 0): 'Clean', (1, 0): 'Clean'} | posição: (1, 0) | desempenho: -1
Passo 3
<Agent> perceives ((1, 0), 'Clean') and does NoOp
Estado técnico: {(0, 0): 'Clean', (1, 0): 'Clean'} | posição: (1, 0) | desempenho: -1
Passo 4
<Agent> perceives ((1, 0), 'Clean') and does NoOp
Estado técnico: {(0, 0): 'Clean', (1, 0): 'Clean'} | posição: (1, 0) | desempenho: -1
O agente terminou na posição (1, 0), com desempenho -1. No final, o ambiente ficou assim: (0, 0): Clean, (1, 0): Clean.


In [ ]:
def executar_agente(factory, estado_inicial, passos=5):
    ambiente = TrivialVacuumEnvironment()
    ambiente.status = dict(estado_inicial)
    agente = factory()
    ambiente.add_thing(agente, loc_A)
    ambiente.run(passos)
    return ambiente.status, agente.location, agente.performance

cenario = {loc_A: 'Clean', loc_B: 'Clean'}

resultado_tabela = executar_agente(TableDrivenVacuumAgent, cenario)
resultado_reflexo = executar_agente(ReflexVacuumAgent, cenario)
resultado_modelo = executar_agente(ModelBasedVacuumAgent, cenario)

print('TableDrivenVacuumAgent:', resultado_tabela)
print(traduzir_saida_tecnica_em_portugues(resultado_tabela))
print()
print('ReflexVacuumAgent:', resultado_reflexo)
print(traduzir_saida_tecnica_em_portugues(resultado_reflexo))
print()
print('ModelBasedVacuumAgent:', resultado_modelo)
print(traduzir_saida_tecnica_em_portugues(resultado_modelo))


### Etapa 5. Comparar as três abordagens

A leitura conceitual da progressão é a seguinte:

1. O agente tabelado precisa consultar uma tabela indexada pelo histórico inteiro de percepções.
2. O agente reflexo decide apenas a partir do percepto atual, o que o torna mais simples.
3. O agente baseado em modelo adiciona memória útil, sem depender de uma tabela gigante de históricos.

Esse é o ganho principal do tutorial: sair de uma solução inviável em escala e chegar a uma solução mais compacta e inteligente.


## Conclusão

O tutorial mostrou a transição de `TableDrivenAgentProgram` até `ModelBasedVacuumAgent`. A principal lição é que agentes melhores não surgem apenas por adicionar mais regras, mas por escolher uma representação mais adequada para decidir.

Em custo de representação, o agente tabelado é o mais caro, porque a tabela cresce com o número de históricos possíveis. O agente reflexo é mais econômico, mas pode falhar quando o percepto atual não é suficiente. O agente baseado em modelo introduz um pequeno custo extra de memória interna, mas costuma ser mais eficiente em comportamento, porque evita ações desnecessárias quando já conhece o estado do ambiente.
